# Day 1+ · Brief → Bubble Diagram → Filled Plan

This extends `bubble_diagram.ipynb`, which stays unchanged. The new parts:

1. **Pick a footprint:** a shape (Rectangle, Square, L-shape, U-shape) and its dimensions.
2. The LLM writes the room programme **sized to that footprint**.
3. The rooms are **packed into the footprint so they fill it completely**, like a first-pass floor plan.
4. An **adjacency score** checks how many of the requested adjacencies actually share a wall.

```
brief + footprint ──► LLM ──► programme ──► graph ──► room order ──► pack into footprint ──► plan + bubble diagram
```

**Run order:** run all cells top to bottom, then use the controls in **Step 9**.

In [ ]:
%pip install -q ollama networkx matplotlib ipywidgets

## Part A · From Day 1 (copied, with one change)

These are the same data shapes, `clean`, `to_graph` and `connect` as `bubble_diagram.ipynb`.

**One change:** `get_program` now also receives the **total floor area**, so the LLM sizes the rooms to fit the footprint. Results are **cached** (`lru_cache`), so trying another shape with the same area doesn't call the LLM again.

In [ ]:
from functools import lru_cache
from typing import Literal

import ollama
import networkx as nx
from pydantic import BaseModel

class Room(BaseModel):
    name: str
    area_m2: float
    zone: Literal["public", "private", "service", "work"]

class Link(BaseModel):
    a: str
    b: str
    strength: Literal["strong", "weak"]

class Program(BaseModel):
    rooms: list[Room]
    adjacencies: list[Link]

SYSTEM = """You are an architectural programmer. Given a brief and a total floor area,
list the rooms with realistic areas in m², a zone for each, and which rooms should be adjacent.
Room areas (including the hall/corridor) must add up to roughly the total floor area.
Use the exact room names in the adjacencies. Include circulation (hall/corridor).
Every room must connect to at least one other room, and the hall must connect to
the living room, so the diagram is one connected graph."""

@lru_cache(maxsize=32)
def get_program(brief: str, total_area: float) -> Program:
    r = ollama.chat(
        model="qwen2.5:14b",
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": f"Brief: {brief}\nTotal floor area: {total_area:.0f} m²"}],
        format=Program.model_json_schema(),
        options={"temperature": 0},
    )
    return Program.model_validate_json(r.message.content)

def clean(p: Program) -> Program:
    names = {r.name for r in p.rooms}
    p.adjacencies = [l for l in p.adjacencies
                     if l.a in names and l.b in names and l.a != l.b]
    return p

def to_graph(p: Program) -> nx.Graph:
    G = nx.Graph()
    for r in p.rooms:
        G.add_node(r.name, area=r.area_m2, zone=r.zone)
    for l in p.adjacencies:
        G.add_edge(l.a, l.b, weight=3 if l.strength == "strong" else 1)
    return G

def find_hub(G: nx.Graph) -> str:
    """The hall/corridor if there is one, otherwise the biggest room."""
    return next((n for n in G if any(w in n.lower() for w in ("hall", "corridor", "circulation"))),
                max(G, key=lambda n: G.nodes[n]["area"]))

def connect(G: nx.Graph) -> nx.Graph:
    hub = find_hub(G)
    for comp in list(nx.connected_components(G)):
        if hub not in comp:
            biggest = max(comp, key=lambda n: G.nodes[n]["area"])
            G.add_edge(hub, biggest, weight=1)
    return G

## Part B · New: footprint → filled plan

### Step 5 · Footprint shapes

A footprint is stored as a **list of rectangles** that together make the shape. Everything else only has to know how to fill a rectangle.

| Shape | Made of | Area |
|---|---|---|
| Rectangle | 1 rect, W × D | W·D |
| Square | 1 rect, W × W (depth ignored) | W² |
| L-shape | bottom strip W × D/2 + left arm W/2 × D/2 | 0.75·W·D |
| U-shape | left arm + bottom strip + right arm (each 1/3 wide or deep) | ≈0.78·W·D |

The rectangles are listed **in walking order** (e.g. left arm → bottom → right arm), so rooms that sit next to each other in the room order also land next to each other in the plan. `outline` gives the polygon used to draw the thick outer wall.

In [ ]:
from dataclasses import dataclass

@dataclass
class Rect:
    x: float
    y: float
    w: float
    h: float

    @property
    def area(self) -> float:
        return self.w * self.h

SHAPES = ["Rectangle", "Square", "L-shape", "U-shape"]

def footprint(shape: str, W: float, D: float) -> list[Rect]:
    if shape == "Rectangle":
        return [Rect(0, 0, W, D)]
    if shape == "Square":
        return [Rect(0, 0, W, W)]
    if shape == "L-shape":
        return [Rect(0, D / 2, W / 2, D / 2),      # arm (top-left)
                Rect(0, 0, W, D / 2)]              # base (bottom)
    if shape == "U-shape":
        return [Rect(0, D / 3, W / 3, 2 * D / 3),          # left arm
                Rect(0, 0, W, D / 3),                      # base
                Rect(2 * W / 3, D / 3, W / 3, 2 * D / 3)]  # right arm
    raise ValueError(shape)

def outline(shape: str, W: float, D: float) -> list[tuple[float, float]]:
    if shape == "Square":
        D = W
    if shape == "L-shape":
        return [(0, 0), (W, 0), (W, D / 2), (W / 2, D / 2), (W / 2, D), (0, D)]
    if shape == "U-shape":
        return [(0, 0), (W, 0), (W, D), (2 * W / 3, D), (2 * W / 3, D / 3),
                (W / 3, D / 3), (W / 3, D), (0, D)]
    return [(0, 0), (W, 0), (W, D), (0, D)]

### Step 6 · Put the rooms in a sensible order

Packing works on a **list** of rooms, so the order decides who ends up next to whom.

- Walk the graph **depth-first**, always following the **strongest** link first. Strongly-linked rooms (e.g. Kitchen → Living) end up consecutive in the list.
- Then move the **hall to the middle** of the list. It's the connector, so it should sit between the two halves of the plan, not at one end.

This is a heuristic, not an optimiser. It's simple, fast and good enough for a first pass.

In [ ]:
def room_order(G: nx.Graph) -> list[str]:
    hub = find_hub(G)
    start = max((n for n in G if n != hub), key=lambda n: G.nodes[n]["area"], default=hub)
    order, seen = [], set()

    def visit(n):
        seen.add(n)
        order.append(n)
        for m in sorted(G[n], key=lambda m: -G[n][m]["weight"]):
            if m not in seen:
                visit(m)

    visit(start)
    for n in G:                      # anything unreachable (shouldn't happen after connect)
        if n not in seen:
            visit(n)
    if hub in order and len(order) > 2:
        order.remove(hub)
        order.insert(len(order) // 2, hub)
    return order

### Step 7 · Fill the footprint

1. **Scale:** the LLM's areas never add up exactly, so every room is scaled by the same factor so that the total equals the footprint area. Proportions stay the same and the box is filled with no gaps.
2. **Share out between the rectangles** (for L and U shapes): walk the ordered list and give each room to the rectangle where its midpoint falls along the cumulative area. Each rectangle gets at least one room.
3. **Split recursively** (an *ordered treemap*): cut the room list into two groups of roughly equal area, then cut the rectangle across its **longer side** in that same proportion. Repeat for each half until each piece holds one room. Cutting the long side keeps rooms from becoming thin slivers.

Because the cuts follow the list order, neighbours in the list become neighbours in the plan.

In [ ]:
def split_rect(names: list[str], areas: dict, r: Rect, placed: dict):
    if len(names) == 1:
        placed[names[0]] = r
        return
    total = sum(areas[n] for n in names)
    # find the cut index that splits the area closest to 50/50
    best_i, best_diff, acc = 1, float("inf"), 0.0
    for i in range(1, len(names)):
        acc += areas[names[i - 1]]
        if abs(acc - total / 2) < best_diff:
            best_i, best_diff = i, abs(acc - total / 2)
    left, right = names[:best_i], names[best_i:]
    f = sum(areas[n] for n in left) / total
    if r.w >= r.h:   # cut across the longer side
        a, b = Rect(r.x, r.y, r.w * f, r.h), Rect(r.x + r.w * f, r.y, r.w * (1 - f), r.h)
    else:
        a, b = Rect(r.x, r.y, r.w, r.h * f), Rect(r.x, r.y + r.h * f, r.w, r.h * (1 - f))
    split_rect(left, areas, a, placed)
    split_rect(right, areas, b, placed)


def layout_plan(G: nx.Graph, rects: list[Rect]) -> dict[str, Rect]:
    order = room_order(G)
    areas = {n: G.nodes[n]["area"] for n in order}

    # 1. share rooms between the footprint rectangles by cumulative area
    fp_total = sum(r.area for r in rects)
    room_total = sum(areas.values())
    bounds, acc = [], 0.0
    for r in rects:
        acc += r.area / fp_total
        bounds.append(acc)
    groups = [[] for _ in rects]
    cum = 0.0
    for n in order:
        mid = (cum + areas[n] / 2) / room_total
        i = next(i for i, b in enumerate(bounds) if mid <= b + 1e-9)
        groups[i].append(n)
        cum += areas[n]

    # every rectangle needs at least one room (borrow from a neighbour)
    for i, g in enumerate(groups):
        if not g:
            donors = [j for j in (i - 1, i + 1) if 0 <= j < len(groups) and len(groups[j]) > 1]
            if donors:
                j = max(donors, key=lambda j: len(groups[j]))
                g.append(groups[j].pop() if j < i else groups[j].pop(0))

    # 2. fill each rectangle (split_rect works with proportions, so this also scales areas to fit)
    placed = {}
    for g, r in zip(groups, rects):
        if g:
            split_rect(g, areas, r, placed)
    return placed

### Step 8 · Check adjacencies and draw

- **`shares_wall`:** two rooms are truly adjacent if they share a wall at least 0.9 m long (wide enough for a door).
- **Adjacency score:** the share of the LLM's requested adjacencies that the packed plan actually satisfies.
- In the plan, **met adjacencies are solid dark lines** and **missed ones are red dashed lines**, so you can see at a glance where the layout disagrees with the programme.
- The final figure shows the **bubble diagram (left)** next to the **filled plan (right)**.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Polygon, Rectangle

COLORS = {"public": "#F4A261", "private": "#2A9D8F",
          "service": "#B8B8B8", "work": "#E76F51"}

def shares_wall(a: Rect, b: Rect, min_len: float = 0.9, tol: float = 1e-4) -> bool:
    if abs(a.x + a.w - b.x) < tol or abs(b.x + b.w - a.x) < tol:      # vertical wall
        if min(a.y + a.h, b.y + b.h) - max(a.y, b.y) >= min_len:
            return True
    if abs(a.y + a.h - b.y) < tol or abs(b.y + b.h - a.y) < tol:      # horizontal wall
        if min(a.x + a.w, b.x + b.w) - max(a.x, b.x) >= min_len:
            return True
    return False


def draw_bubbles(ax, G: nx.Graph, title: str, scale: float = 150):
    pos = nx.spring_layout(G, weight="weight", seed=42, k=1.2)
    strong = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] == 3]
    weak = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] == 1]
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=strong, width=2.5, edge_color="#333")
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=weak, width=1, style="dashed", edge_color="#999")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[G.nodes[n]["area"] * scale for n in G],
                           node_color=[COLORS[G.nodes[n]["zone"]] for n in G], alpha=1)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=7,
                            labels={n: f"{n}\n{G.nodes[n]['area']:.0f} m²" for n in G})
    ax.set_title(title, fontsize=11)
    ax.axis("off")


def draw_plan(ax, G: nx.Graph, placed: dict, shape: str, W: float, D: float) -> float:
    if shape == "Square":
        D = W
    # rooms
    for n, r in placed.items():
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, facecolor=COLORS[G.nodes[n]["zone"]],
                               edgecolor="white", linewidth=2.5))
        fs = 7 if min(r.w, r.h) > 1.5 else 5.5
        ax.text(r.x + r.w / 2, r.y + r.h / 2, f"{n}\n{r.area:.1f} m²",
                ha="center", va="center", fontsize=fs,
                rotation=90 if r.h > 2 * r.w and r.w < 2 else 0)
    # adjacency check lines
    met = total = 0
    for u, v, d in G.edges(data=True):
        if u not in placed or v not in placed:
            continue
        a, b = placed[u], placed[v]
        ok = shares_wall(a, b)
        total += 1
        met += ok
        ax.plot([a.x + a.w / 2, b.x + b.w / 2], [a.y + a.h / 2, b.y + b.h / 2],
                color="#222" if ok else "#D62828", linestyle="-" if ok else "--",
                linewidth=1.2 if ok else 1.0, alpha=0.6, zorder=3)
    score = met / total if total else 1.0
    # outer wall + dimensions
    ax.add_patch(Polygon(outline(shape, W, D), closed=True, fill=False,
                         edgecolor="#111", linewidth=4, zorder=4))
    ax.annotate(f"{W:g} m", (W / 2, -0.6), ha="center", va="top", fontsize=9)
    ax.annotate(f"{D:g} m", (-0.6, D / 2), ha="right", va="center", fontsize=9, rotation=90)
    ax.set_xlim(-1.5, W + 0.5)
    ax.set_ylim(-1.5, D + 0.5)
    ax.set_aspect("equal")
    ax.axis("off")
    fp_area = sum(r.area for r in footprint(shape, W, D))
    ax.set_title(f"{shape} · {W:g}×{D:g} m · {fp_area:.0f} m²   |   adjacencies met: {met}/{total} ({score:.0%})",
                 fontsize=11)
    return score


def run(brief: str, shape: str = "Rectangle", W: float = 12, D: float = 10, out: str = "plan.png"):
    rects = footprint(shape, W, D)
    total_area = round(sum(r.area for r in rects))
    program = get_program(brief, float(total_area))
    G = connect(to_graph(clean(program)))
    placed = layout_plan(G, rects)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7.5),
                                   gridspec_kw={"width_ratios": [1, 1.1]})
    draw_bubbles(ax1, G, "Bubble diagram (from the LLM)")
    draw_plan(ax2, G, placed, shape, W, D)
    fig.legend(handles=[Patch(color=c, label=z) for z, c in COLORS.items()],
               loc="lower center", ncol=4, frameon=False)
    fig.suptitle(brief, fontsize=14)
    plt.savefig(out, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    return program, G, placed

## Step 9 · Pick a footprint and generate

Type a brief, choose a shape and dimensions, then click **Generate**.

- The **first** click for a new brief or area calls the LLM (10–30 s). After that, switching between shapes with the same area is instant, thanks to the cache.
- **Square** ignores *Depth*.
- Each run saves `plan.png` in this folder.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

brief_in = w.Text(value="2-bed flat with a home studio", description="Brief",
                  layout=w.Layout(width="520px"))
shape_in = w.ToggleButtons(options=SHAPES, value="Rectangle", description="Shape")
width_in = w.Dropdown(options=[8, 10, 12, 15, 18], value=12, description="Width (m)")
depth_in = w.Dropdown(options=[6, 8, 10, 12, 15], value=10, description="Depth (m)")
go = w.Button(description="Generate", button_style="primary", icon="play")
out = w.Output()

def _on_shape(change):
    depth_in.disabled = change["new"] == "Square"
shape_in.observe(_on_shape, names="value")

def _on_go(_):
    with out:
        clear_output(wait=True)
        print("Generating… (first run per brief calls the LLM)")
        run(brief_in.value, shape_in.value, width_in.value, depth_in.value)

go.on_click(_on_go)
display(w.VBox([brief_in, shape_in, w.HBox([width_in, depth_in]), go, out]))

### No widgets? Same thing in one line

If the controls don't appear (or for a quick batch), call `run` directly:

In [ ]:
_ = run("2-bed flat with a home studio", shape="L-shape", W=12, D=10, out="plan_L.png")

## Learnings

- Treating a footprint as **a list of rectangles** turns complex shapes into one simple problem: fill a rectangle.
- An **ordered treemap** is a cheap way to turn a graph into a plan, but it only respects adjacency *approximately*. The **adjacency score** makes that honest and measurable.
- **Caching** the LLM call keeps the slow step out of the design loop.

## What I'd do with more time (but didn't)

- Optimise the room order to **maximise the adjacency score** (try many orders and keep the best, or use a genetic algorithm, a natural next step with Wallacei in Grasshopper).
- Enforce **minimum room widths** (no 1.2 m bedrooms) and keep halls as corridors.
- Doors on shared walls, windows on the outer wall, and correct orientation.
- Send the rectangles to **Rhino/Grasshopper** (via Hops or the Rhino MCP) to extrude a massing model.